# Incremental 12–14시 LSTM + 14시 CatBoost 앙상블

- LSTM: `shortterm_long_2019to2025.csv`의 13시점×16채널과 결측 마스크
- CatBoost: `final_train_dataset_19to25_master.csv`의 기존 14시 표
- 분할: 2019–2023 학습 / 2024 조기종료·TA/HM별 가중치 선택 / 2025 최종 평가
- 점수: `RMSE(TA) + 0.1 × RMSE(HM)`

2025 라벨은 하이퍼파라미터나 앙상블 가중치 선택에 사용하지 않습니다.

In [ ]:
# 1. Drive 마운트
import os
from pathlib import Path
from google.colab import drive

candidates = [Path('/content/gdrive_sme_model'), Path('/content/gdrive_sme_model_2')]
mounted = next((p for p in candidates if os.path.ismount(p)), None)
if mounted is not None:
    MOUNT_POINT = mounted
else:
    MOUNT_POINT = next((p for p in candidates if not p.exists() or not any(p.iterdir())), None)
    if MOUNT_POINT is None:
        raise RuntimeError('마운트 후보 폴더가 비어 있지 않습니다. 런타임을 재시작하세요.')
    drive.mount(str(MOUNT_POINT), force_remount=False)
print('MOUNT_POINT =', MOUNT_POINT)

In [ ]:
# 2. 저장소와 데이터 경로
import subprocess, sys
BRANCH = 'agent/shortterm-12to14-pipeline'
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_DIR = Path('/content/SME_DATA_incremental_model')
BASE = MOUNT_POINT / 'MyDrive/SME_DATA/processed_station_features'
MASTER_CSV = BASE / 'final_train_dataset_19to25_master.csv'
SHORTTERM_CSV = BASE / 'shortterm_12to14_data/incremental_12to14_tables/shortterm_long_2019to2025.csv'
OUTPUT_DIR = BASE / 'model_experiments_19to25/incremental_lstm_catboost_ensemble'
for path in [MASTER_CSV, SHORTTERM_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)
print('MASTER_CSV =', MASTER_CSV)
print('SHORTTERM_CSV =', SHORTTERM_CSV)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# 3. 최신 코드와 패키지 준비
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'], check=True)
import torch
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())

In [ ]:
# 4. 학습 실행
def run_live(cmd):
    print('$', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(
        list(map(str, cmd)), cwd=REPO_DIR, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode:
        raise RuntimeError(f'학습 실패 (exit code {returncode})')

cmd = [
    sys.executable, '-u', 'scripts/train_incremental_lstm_catboost_ensemble.py',
    '--master-csv', MASTER_CSV, '--shortterm-long-csv', SHORTTERM_CSV,
    '--output-dir', OUTPUT_DIR, '--device', 'auto', '--threads', 4,
]
run_live(cmd)

In [ ]:
# 5. 성능과 모델 영향 확인
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUTPUT_DIR / 'metrics.csv'))
display(pd.read_csv(OUTPUT_DIR / 'ensemble_component_weights.csv'))
display(pd.read_csv(OUTPUT_DIR / 'test_missingness_subgroup_metrics.csv'))

In [ ]:
# 6. TA/HM별 원본 피처 중요도
cat_imp = pd.read_csv(OUTPUT_DIR / 'catboost_feature_importance.csv')
lstm_imp = pd.read_csv(OUTPUT_DIR / 'lstm_permutation_importance.csv')
for target in ['TA', 'HM']:
    print('\nCatBoost', target)
    display(cat_imp[cat_imp.target == target].sort_values('rank').head(15))
    print('LSTM permutation', target)
    display(lstm_imp[lstm_imp.target == target].sort_values('rank').head(15))

In [ ]:
# 7. 전처리 산출물 간 채널 일관성 진단
consistency = pd.read_csv(OUTPUT_DIR / 'master_shortterm_feature_consistency.csv')
display(consistency.sort_values(['year', 'correlation']).groupby('year').head(4))